# 11. Staggered axes

An `Axis` has two classes of cells: its points, and the points halfway between them. On ICON's vertical axis, the full levels `K` are the midpoints of the layers, and the half levels `Staggered[K]` are the interfaces between the layers: `nlev` full levels and `nlev + 1` half levels. Some fields live on the full levels, e.g. the temperature, others on the half levels, e.g. the vertical wind.

Three full levels, from the top down, and their four interfaces, at their positions along the axis:

```
  Staggered[K](0) ─────────  -1/2   (top interface)
          K(0)                 0
  Staggered[K](1) ─────────   1/2
          K(1)                 1
  Staggered[K](2) ─────────   3/2
          K(2)                 2
  Staggered[K](3) ─────────   5/2   (bottom interface)
```

Contents: staggered dimensions · half offsets · operators on half levels.

## Staggered dimensions

| Concept | Kind | Meaning |
|---|---|---|
| `Staggered[D]` | dimension | The staggered partner of a declared `Axis` `D`, itself an `Axis`, at the half-integer positions of `D`: `Staggered[K](i)` is at position `i - 1/2`. One class per axis, created on first use |
| `flip_staggered` | function | `flip_staggered(D)` is the partner of `D`: `Staggered[D]` for a declared axis, `D.base` for a staggered one. A `TypeError` for the other kinds |
| `base` | attribute | The declared axis of a staggered one: `Staggered[K].base` is `K` |

- Extents are declared, never derived: `nlev` full levels and `nlev + 1` half levels are two numbers, e.g. the domains `K[0:3]` and `Staggered[K][0:4]`.
- A domain never has both `K` and `Staggered[K]`: they are the two cell classes of one axis (a `ValueError`).
- `Staggered[Staggered[K]]` is `K` at runtime. Write `K`: type checkers see `Staggered[Staggered[K]]` as another type.

In [1]:
import contextlib
import io

from concepts import Axis, Dims, Field, Location, Staggered, check_domain, flip_staggered
from implementation import NumPyArrayField


class Cell(Location): ...


class K(Axis): ...  # the full levels, from the top down


S = Staggered[K]  # the half levels, the interfaces: Staggered[K](i) is at the position i - 1/2
nlev = 3  # full levels, so nlev + 1 = 4 interfaces: the extents are declared, K[0:3] and S[0:4]
print(
    "S:",
    S.__name__,
    "| base:",
    S.base.__name__,
    "| Staggered[S] is K:",
    Staggered[S] is K,
    "| flip_staggered(K) is S:",
    flip_staggered(K) is S,
    "| an Axis:",
    issubclass(S, Axis),
)
# Staggered[K](1) is at the position 1 - 1/2
print("points:", K.at(1), S.at(1), "| interned:", Staggered[K] is S)
try:
    Staggered[Cell]  # a `Location` has no staggered partner
except TypeError as error:
    print("TypeError:", error)
try:
    check_domain({K: range(nlev), S: range(nlev + 1)})
except ValueError as error:
    print("ValueError:", error)
try:
    check_domain({S: (0, 2)})  # an `Axis`, staggered or not, has ranges only
except ValueError as error:
    print("ValueError:", error)

S: Staggered[K] | base: K | Staggered[S] is K: True | flip_staggered(K) is S: True | an Axis: True
points: K(1) Staggered[K](1) | interned: True
TypeError: 'Cell' has no staggered partner: only an `Axis` has one.
ValueError: A domain cannot have both 'K' and 'Staggered[K]': they are the two cell classes of one axis.
ValueError: Index sets are not allowed along 'Staggered[K]'.


## Half offsets

| Concept | Kind | Meaning |
|---|---|---|
| `HalfOffset` | alias | `tuple[Dimension, float]`: an offset to the staggered partner, written `K + 0.5` or `K - 0.5` |

`f(K + 0.5)` premaps a field `f` along `Staggered[K]` to a field along `K`, and `f(Staggered[K] + 0.5)` one along `K` to one along `Staggered[K]`. The data is unchanged: the dimension is renamed, and its domain shifted. For an offset `h`, the result at the position `x` is the value of `f` at the position `x + h`: at `D(i)`, the value of `f` at `flip_staggered(D)(i + c)`, with `c = h + 1/2` if `D` is a declared axis and `c = h - 1/2` if `D` is staggered. With `w` on the half levels and `t` on the full levels:

| Read | At | Reads |
|---|---|---|
| `w(K - 0.5)` | level `i` | interface `i`, above it |
| `w(K + 0.5)` | level `i` | interface `i + 1`, below it |
| `t(Staggered[K] + 0.5)` | interface `i` | level `i`, below it |
| `t(Staggered[K] - 0.5)` | interface `i` | level `i - 1`, above it |

Integer offsets along `Staggered[K]` are shifts, as along any `Axis`. Other offsets, e.g. `K + 0.3`, raise a `ValueError`, and a half offset of a field without the partner, e.g. `t(K + 0.5)`, a `TypeError`.

In [2]:
t = NumPyArrayField({K: range(nlev)}, [1.0, 4.0, 9.0])  # on the full levels
w = NumPyArrayField({S: range(nlev + 1)}, [0.0, 10.0, 20.0, 40.0])  # on the half levels

print("K - 0.5 == (K, -0.5):", K - 0.5 == (K, -0.5), "| K + 1 == (K, 1):", K + 1 == (K, 1))
# f(D + h) at the position x is f at the position x + h: at D(i), f at flip_staggered(D)(i + c),
# with c = h + 1/2 along a declared axis D, and c = h - 1/2 along a staggered one:
# K(i) reads Staggered[K](i): c = -0.5 + 1/2 = 0, the domain renamed (K(3) is below the bottom)
print("w(K - 0.5) =", w(K - 0.5))
# K(i) reads Staggered[K](i + 1): c = 0.5 + 1/2 = 1, the domain shifted by -1
print("w(K + 0.5) =", w(K + 0.5))
# Staggered[K](i) reads K(i): c = 0.5 - 1/2 = 0
print("t(S + 0.5) =", t(S + 0.5))
# Staggered[K](i) reads K(i - 1): c = -0.5 - 1/2 = -1, the domain shifted by +1
print("t(S - 0.5) =", t(S - 0.5))
print("w(S + 1)   =", w(S + 1))  # a shift: Staggered[K](i) reads Staggered[K](i + 1)
try:
    K + 0.3
except ValueError as error:
    print("ValueError:", error)
try:
    t(K + 0.5)  # t is on the full levels already
except TypeError as error:
    print("TypeError:", error)

K - 0.5 == (K, -0.5): True | K + 1 == (K, 1): True
w(K - 0.5) = Field(K[0:4]: [0.0, 10.0, 20.0, 40.0])
w(K + 0.5) = Field(K[-1:3]: [0.0, 10.0, 20.0, 40.0])
t(S + 0.5) = Field(Staggered[K][0:3]: [1.0, 4.0, 9.0])
t(S - 0.5) = Field(Staggered[K][1:4]: [1.0, 4.0, 9.0])
w(S + 1)   = Field(Staggered[K][-1:3]: [0.0, 10.0, 20.0, 40.0])
ValueError: 'K + 0.3': an offset is an integer, or a half-integer to the staggered partner, e.g. `K + 0.5`.
TypeError: `(K, 0.5)` reads a field along 'Staggered[K]', not K[0:3].


## Operators on half levels

Half offsets are reads like the others, in field operators on every backend: e.g. `interp` interpolates from the interfaces to the levels, and `grad` takes the differences of the levels across the interfaces. The `Extents` hold them as steps, `(K, -0.5)`, and domains propagate through them:

- **forward**, `forward_domain`: from the domain of `w`, `Staggered[K][0:4]`, the levels with both of their interfaces, `K[0:3]`;
- **backward**, `backward_domain`: from a requested level, `K[1:2]`, the interfaces it needs, `Staggered[K][1:3]`; `lazy_backward` computes only the requested level, reading only those interfaces.

Fields on columns of cells, `(Cell, Staggered[K])`, are premapped along their `Staggered[K]` dimension only, and scan operators run along `Staggered[K]` like along any `Axis`.

In [3]:
from implementation import (
    backward_domain,
    column_major,
    compiled,
    eager,
    field_operator,
    format_domain,
    forward_domain,
    lazy,
    lazy_backward,
)


@field_operator()
def interp(w: Field[Dims[Staggered[K]], float]) -> Field[Dims[K], float]:
    return 0.5 * (w(K - 0.5) + w(K + 0.5))  # the mean of the interfaces above and below


@field_operator()
def grad(t: Field[Dims[K], float]) -> Field[Dims[Staggered[K]], float]:
    return t(Staggered[K] + 0.5) - t(Staggered[K] - 0.5)  # the level below minus the one above


def format_step(step):  # a (half) offset, e.g. `K-0.5`
    return f"{step[0].__name__}{step[1]:+g}"


# levels 0, 1, 2: (0 + 10) / 2 = 5, (10 + 20) / 2 = 15, (20 + 40) / 2 = 30;
# interfaces 1, 2 (the inner ones): 4 - 1 = 3, 9 - 4 = 5
for backend in (None, eager, lazy, lazy_backward, column_major, compiled):
    with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
        values = [
            (op if backend is None else backend(op))(f).data.tolist()
            for op, f in ((interp, w), (grad, t))
        ]
    print(f"{str(backend or 'embedded'):<14}", "interp:", values[0], "| grad:", values[1])

reads = interp.__gt_extents__["w"]
print("extents of interp:", ["(" + ", ".join(map(format_step, read)) + ")" for read in reads])
# forward: Staggered[K][0:4] gives K[0:4] (c = 0) and K[-1:3] (c = 1), intersected:
# K[0:4] ∩ K[-1:3] = K[0:3]
print("forward: ", format_domain(forward_domain({S: range(0, 4)}, reads)))
# backward: level 1 reads the interfaces 1 (c = 0) and 2 (c = 1)
print("backward:", format_domain(backward_domain({K: range(1, 2)}, reads)))
level_1 = lazy_backward(interp)[{K: range(1, 2)}](w)
values = level_1.data  # computed on first use: the backend prints the computed domain
print("interp[K[1:2]](w) =", level_1)  # (10 + 20) / 2 = 15

embedded       interp: [5.0, 15.0, 30.0] | grad: [3.0, 5.0]
eager          interp: [5.0, 15.0, 30.0] | grad: [3.0, 5.0]
lazy           interp: [5.0, 15.0, 30.0] | grad: [3.0, 5.0]
lazy_backward  interp: [5.0, 15.0, 30.0] | grad: [3.0, 5.0]
column_major   interp: [5.0, 15.0, 30.0] | grad: [3.0, 5.0]
compiled       interp: [5.0, 15.0, 30.0] | grad: [3.0, 5.0]
extents of interp: ['(K-0.5)', '(K+0.5)']
forward:  K[0:3]
backward: Staggered[K][1:3]
[lazy_backward] computed K[1:2]
interp[K[1:2]](w) = Field(K[1:2]: [15.0])


In [4]:
from implementation import scan_operator

# two columns of cells, with 4 interfaces each
w2 = NumPyArrayField({Cell: range(2), S: range(4)}, [[0.0, 10.0, 20.0, 40.0], [2.0, 4.0, 6.0, 8.0]])


@field_operator()
def interp2(w: Field[Dims[Cell, Staggered[K]], float]) -> Field[Dims[Cell, K], float]:
    return 0.5 * (w(K - 0.5) + w(K + 0.5))


# cell 0: as above, 5, 15, 30; cell 1: (2 + 4) / 2 = 3, (4 + 6) / 2 = 5, (6 + 8) / 2 = 7
print("interp2(w2) =", interp2(w2))
with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
    same = [
        backend(interp2)(w2).data.tolist() == interp2(w2).data.tolist()
        for backend in (eager, lazy, lazy_backward, column_major, compiled)
    ]
print("the same on the 5 backends:", same)


@scan_operator(axis=Staggered[K])
def accumulated(above: float, x: float) -> float:
    return above + x  # the sum over the interfaces from the top


print("accumulated(w) =", accumulated(w))  # 0, 0 + 10, 10 + 20, 30 + 40 = 0, 10, 30, 70

interp2(w2) = Field(Cell[0:2], K[0:3]: [[5.0, 15.0, 30.0], [3.0, 5.0, 7.0]])
the same on the 5 backends: [True, True, True, True, True]
accumulated(w) = Field(Staggered[K][0:4]: [0.0, 10.0, 30.0, 70.0])


## Summary

| Feature | What is checked |
|---|---|
| `Staggered[D]`, `flip_staggered` | only an `Axis` has a staggered partner (a `TypeError`); `Staggered[Staggered[K]]` is `K` at runtime |
| Domains | never both `K` and `Staggered[K]` (a `ValueError`); along `Staggered[K]`, ranges only, as along any `Axis` |
| Half offsets, `K + 0.5` | a half-integer (else a `ValueError`); the field has the partner dimension (else a `TypeError`) |
| Operators on half levels | the same values on every backend; extents with half offsets; forward and backward propagation |
| Scans along `Staggered[K]` | as along any `Axis` |